# 03 — Custom Training (Colab GPU) + Model Evaluation

**Capstone deliverables** — *Custom Data & Training (15 pts)* and *Model Evaluation (25 pts)*.

## How to run this notebook
1. Upload to Google Colab (or open from GitHub).
2. **Runtime → Change runtime type → T4 GPU** (training is not feasible on CPU).
3. **Runtime → Run all**. Expected total time ≈ **2–2.5 h** (Run A ≈ 45–60 min, Run B ≈ 60–75 min,
   evaluation ≈ 10 min).
4. When finished: **File → Download → Download .ipynb** (keeps outputs) and put it in the repo's
   `notebooks/` folder; download `best.pt` (cell at the bottom zips all artifacts).

No Kaggle API key is required — the dataset is public and downloads anonymously via `kagglehub`.

In [ ]:
%pip install -q ultralytics kagglehub

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 1 — The dataset: Construction Site Safety (Roboflow → Kaggle)

| | |
|---|---|
| Source | Kaggle: [`snehilsanyal/construction-site-safety-image-dataset-roboflow`](https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow) (originally Roboflow Universe *Construction Site Safety*) |
| License | CC BY 4.0 |
| Size | 2,641 images (train 2,445 / valid 114 / test 82) — YOLO format |
| Classes | 10 PPE classes (below) |

This is a **real public PPE dataset** (not coco8) with paired compliance classes:
`Hardhat` vs `NO-Hardhat`, `NO-Safety Vest` vs `Safety Vest` — exactly what a compliance
monitor needs, because a violation is a *positive detection* of `NO-*`, not the absence of a box.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('snehilsanyal/construction-site-safety-image-dataset-roboflow')
print('downloaded to:', path)
DATA_ROOT = Path(path) / 'css-data'
print('subdirs:', sorted(d.name for d in DATA_ROOT.iterdir() if d.is_dir()))

In [ ]:
import matplotlib.pyplot as plt
import yaml
from collections import Counter
from pathlib import Path

NAMES = {
    0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest',
    5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'vehicle',
}

DATA_YAML = Path('/content/css_data.yaml')
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(DATA_ROOT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': NAMES,
}))
print(DATA_YAML.read_text())

for split in ('train', 'valid', 'test'):
    n = len(list((DATA_ROOT / split / 'images').glob('*')))
    print(f'{split}: {n} images')

cnt = Counter()
for lf in (DATA_ROOT / 'train' / 'labels').glob('*.txt'):
    for line in lf.read_text().splitlines():
        if line.strip():
            cnt[int(line.split()[0])] += 1
pairs = sorted(cnt.items())
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([NAMES[c] for c, _ in pairs], [n for _, n in pairs], color='#2b8a3e')
ax.set_title('Train label distribution (class imbalance is expected and discussed later)')
ax.set_ylabel('instances')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 2 — Training Run A (baseline)

Fine-tune `yolo11n.pt` (COCO-pretrained, transfer learning) with mostly default knobs:

| knob | value |
|---|---|
| epochs | 20 |
| imgsz | 640 |
| batch | -1 (AutoBatch — fills T4 memory) |
| freeze | 0 (whole network trains) |
| augmentation | Ultralytics defaults (mosaic, hsv, flips…) |
| seed | 42 (reproducibility) |

In [ ]:
from ultralytics import YOLO

model_a = YOLO('yolo11n.pt')
res_a = model_a.train(
    data=str(DATA_YAML), epochs=20, imgsz=640, batch=-1, device=0,
    seed=42, plots=True, project='/content/runs', name='run_a_baseline',
)
RUN_A = Path(getattr(res_a, 'save_dir', '/content/runs/run_a_baseline'))
if not RUN_A.exists():
    RUN_A = sorted(Path('/content/runs').glob('**/weights/best.pt'))[-1].parents[1]
print('Run A saved in:', RUN_A)

In [ ]:
import pandas as pd

df_a = pd.read_csv(RUN_A / 'results.csv')
df_a.columns = [c.strip() for c in df_a.columns]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(df_a['epoch'], df_a['train/box_loss'], label='train'); axes[0].plot(df_a['epoch'], df_a['val/box_loss'], label='val')
axes[0].set_title('box_loss'); axes[0].legend()
axes[1].plot(df_a['epoch'], df_a['train/cls_loss'], label='train'); axes[1].plot(df_a['epoch'], df_a['val/cls_loss'], label='val')
axes[1].set_title('cls_loss'); axes[1].legend()
axes[2].plot(df_a['epoch'], df_a['metrics/mAP50(B)'], label='mAP50'); axes[2].plot(df_a['epoch'], df_a['metrics/mAP50-95(B)'], label='mAP50-95')
axes[2].set_title('mAP'); axes[2].legend()
plt.tight_layout()
plt.show()
df_a[['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(3)

## 3 — Training Run B (tuned: regularisation + backbone freezing)

What we changed **and why**:

| knob | A | B | why |
|---|---|---|---|
| epochs | 20 | **30** (patience **10**) | more headroom, early-stop if val stalls |
| freeze | 0 | **10** | freeze backbone — keep generic COCO features, train PPE-specific head |
| weight_decay | 0.0005 | **0.001** | stronger L2 against memorisation |
| hsv_v | default 0.4 | **0.5** | site lighting varies (shadows, glare) |
| degrees | 0 | **10** | cameras are never perfectly level |
| translate / scale | 0.1 / 0.5 | **0.2 / 0.7** | workers appear at many distances |
| close_mosaic | 10 | **10** | disable mosaic near the end for stable final epochs |

In [ ]:
model_b = YOLO('yolo11n.pt')
res_b = model_b.train(
    data=str(DATA_YAML), epochs=30, imgsz=640, batch=-1, device=0,
    freeze=10, patience=10, weight_decay=0.001,
    hsv_v=0.5, degrees=10.0, translate=0.2, scale=0.7, fliplr=0.5,
    mosaic=1.0, close_mosaic=10,
    seed=42, plots=True, project='/content/runs', name='run_b_tuned',
)
RUN_B = Path(getattr(res_b, 'save_dir', '/content/runs/run_b_tuned'))
if not RUN_B.exists():
    RUN_B = sorted(Path('/content/runs').glob('**/weights/best.pt'))[-2].parents[1]
print('Run B saved in:', RUN_B)

In [ ]:
df_b = pd.read_csv(RUN_B / 'results.csv')
df_b.columns = [c.strip() for c in df_b.columns]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df_a['epoch'], df_a['metrics/mAP50(B)'], 'o-', label='A baseline')
axes[0].plot(df_b['epoch'], df_b['metrics/mAP50(B)'], 's-', label='B tuned')
axes[0].set_title('mAP50'); axes[0].legend()
axes[1].plot(df_a['epoch'], df_a['metrics/mAP50-95(B)'], 'o-', label='A baseline')
axes[1].plot(df_b['epoch'], df_b['metrics/mAP50-95(B)'], 's-', label='B tuned')
axes[1].set_title('mAP50-95'); axes[1].legend()
plt.tight_layout()
plt.show()
print('A best mAP50-95:', df_a['metrics/mAP50-95(B)'].max().round(4))
print('B best mAP50-95:', df_b['metrics/mAP50-95(B)'].max().round(4))

## 4 — Overfitting / underfitting readout

Diagnosis rules (course Part 4):
- **Overfit**: train loss keeps dropping while val loss rises / val mAP stalls.
- **Underfit**: both losses stay high, mAP flat.
- **Healthy**: both losses converge low, val mAP climbs then plateaus.

In [ ]:
for tag, df in (('A', df_a), ('B', df_b)):
    tr = df['train/box_loss']; va = df['val/box_loss']
    last5_tr = tr.tail(5).mean(); first5_va = va.head(5).mean(); last5_va = va.tail(5).mean()
    trend = 'RISING (overfit signal)' if last5_va > first5_va else 'flat/falling (healthy)'
    print(f'Run {tag}: train box_loss last-5 mean = {last5_tr:.4f} | '
          f'val box_loss first-5 = {first5_va:.4f} -> last-5 = {last5_va:.4f} ({trend})')
    print(f'        best mAP50-95 = {df["metrics/mAP50-95(B)"].max():.4f} at epoch {int(df.loc[df["metrics/mAP50-95(B)"].idxmax(), "epoch"])}')

In [ ]:
map_a = df_a['metrics/mAP50-95(B)'].max()
map_b = df_b['metrics/mAP50-95(B)'].max()
BEST_DIR, tag = (RUN_B, 'B') if map_b >= map_a else (RUN_A, 'A')
BEST_PT = BEST_DIR / 'weights' / 'best.pt'
get_ipython().system(f'cp {BEST_PT} /content/best.pt')
print(f'Selected Run {tag} (mAP50-95 {max(map_a, map_b):.4f}) -> /content/best.pt')

## 5 — Evaluation on the validation split (Deliverable: Model Evaluation)

`model.val()` runs inference over `valid/images` (114 unseen site images, 697 instances),
matches predictions to ground truth with IoU, and reports COCO-style metrics.

**Why these metrics matter for THIS use case:**
- A **false negative on `NO-Hardhat`** = an unhelmeted worker we fail to flag = a real
  safety incident walking past the camera. This is the costliest possible error.
- A **false positive** = a compliant worker flagged for inspection = wasted supervisor
  time, but nobody gets hurt.
→ The system must be tuned for **high recall on the `NO-*` classes**, accepting some
extra false alarms. The threshold study below quantifies that trade-off.

In [ ]:
best_model = YOLO('/content/best.pt')
metrics = best_model.val(data=str(DATA_YAML), split='val', iou=0.6, plots=True)

print('=== headline metrics ===')
print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"precision  : {metrics.box.mp:.4f}")
print(f"recall     : {metrics.box.mr:.4f}")

idx = list(metrics.box.ap_class_index)
rows = []
for pos, i in enumerate(idx):
    rows.append({
        'class': metrics.names[int(i)],
        'precision': round(float(metrics.box.p[pos]), 3),
        'recall': round(float(metrics.box.r[pos]), 3),
        'AP50': round(float(metrics.box.ap50[pos]), 3),
        'AP50-95': round(float(metrics.box.ap[pos]), 3),
    })
per_class = pd.DataFrame(rows).sort_values('class')
per_class

In [ ]:
from IPython.display import Image as IPyImage, display

save_dir = Path(metrics.save_dir)
print('val outputs in:', save_dir)
for png in ('confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png'):
    p = save_dir / png
    if p.exists():
        print(f'--- {png} ---')
        display(IPyImage(filename=str(p), width=560))

## 6 — Confidence-threshold study (choosing the operating point)

The detector emits *scores*; **we** choose the cut-off. Lower conf → higher recall
(fewer missed violations) but lower precision (more false alarms). We sweep the
threshold and track **`NO-Hardhat` recall/precision** explicitly, because that class
carries the safety cost.

**IoU choice:** metric mAP50-95 integrates IoU 0.50→0.95 (strict localization quality);
the confusion matrix above matches at IoU 0.60 — a pragmatic duplicate-suppression level
for axis-aligned PPE boxes.

In [ ]:
sweep = []
for conf in (0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 0.90):
    m = best_model.val(data=str(DATA_YAML), split='val', conf=conf, iou=0.6, plots=False, verbose=False)
    mp, mr = float(m.box.mp), float(m.box.mr)
    f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0
    idx = list(m.box.ap_class_index)
    nh = m.names[2]
    pos = idx.index(2) if 2 in idx else None
    nh_p = float(m.box.p[pos]) if pos is not None else 0.0
    nh_r = float(m.box.r[pos]) if pos is not None else 0.0
    sweep.append({'conf': conf, 'P_all': round(mp, 3), 'R_all': round(mr, 3), 'F1_all': round(f1, 3),
                  'NO-Hardhat_P': round(nh_p, 3), 'NO-Hardhat_R': round(nh_r, 3)})
    print(f'conf={conf:.2f} done')
sweep_df = pd.DataFrame(sweep)
sweep_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(sweep_df['conf'], sweep_df['P_all'], 'o-', label='precision (all)')
axes[0].plot(sweep_df['conf'], sweep_df['R_all'], 's-', label='recall (all)')
axes[0].plot(sweep_df['conf'], sweep_df['F1_all'], '^-', label='F1 (all)')
axes[0].set_xlabel('conf threshold'); axes[0].legend(); axes[0].set_title('All classes')
axes[1].plot(sweep_df['conf'], sweep_df['NO-Hardhat_P'], 'o-', label='NO-Hardhat precision')
axes[1].plot(sweep_df['conf'], sweep_df['NO-Hardhat_R'], 's-', label='NO-Hardhat recall')
axes[1].set_xlabel('conf threshold'); axes[1].legend(); axes[1].set_title('NO-Hardhat (the safety-critical class)')
plt.tight_layout()
plt.show()

## 7 — FN vs FP: the business cost of each operating point

Course-style ROI model with **stated assumptions** (adjust to your site):
- ~40 unhelmeted-worker events per day reach the camera.
- **FN cost = 500 SAR** (unmitigated head-injury risk, regulatory exposure).
- **FP cost = 50 SAR** (supervisor walks over, checks, clears the worker).

From per-class precision/recall at each threshold: `TP = R×40`, `FN = 40−TP`,
`FP = TP×(1/P − 1)` → **expected daily cost** per operating point.

In [ ]:
VIOLATIONS_PER_DAY = 40
COST_FN = 500
COST_FP = 50

costs = []
for _, row in sweep_df.iterrows():
    R = row['NO-Hardhat_R']; P = row['NO-Hardhat_P']
    tp = R * VIOLATIONS_PER_DAY
    fn = VIOLATIONS_PER_DAY - tp
    fp = tp * (1 / P - 1) if P > 0 else float('inf')
    costs.append({'conf': row['conf'], 'TP': round(tp, 1), 'FN': round(fn, 1),
                  'FP': round(fp, 1), 'daily_cost_SAR': round(fn * COST_FN + fp * COST_FP)})
cost_df = pd.DataFrame(costs)
best_row = cost_df.loc[cost_df['daily_cost_SAR'].idxmin()]
print(cost_df.to_string(index=False))
print(f"\nLowest-cost operating point: conf={best_row['conf']} "
      f"({best_row['FN']} missed violations/day, {best_row['FP']} false alarms/day)")

In [ ]:
import json

CHOSEN_CONF = float(best_row['conf'])
print('Chosen deployment confidence threshold:', CHOSEN_CONF)
final = best_model.val(data=str(DATA_YAML), split='val', conf=CHOSEN_CONF, iou=0.6, plots=False, verbose=False)
summary = {
    'chosen_conf': CHOSEN_CONF,
    'run_a_best_map5095': round(float(map_a), 4),
    'run_b_best_map5095': round(float(map_b), 4),
    'selected_run': tag,
    'val_mAP50': round(float(metrics.box.map50), 4),
    'val_mAP50-95': round(float(metrics.box.map), 4),
    'val_precision': round(float(metrics.box.mp), 4),
    'val_recall': round(float(metrics.box.mr), 4),
    'expected_daily_cost_SAR': int(best_row['daily_cost_SAR']),
}
with open('/content/metrics.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))

## 8 — Package artifacts for the repo

Run this last cell, then download **two files**: this executed notebook
(*File → Download → Download .ipynb*) and `capstone_artifacts.zip`.
Place them in the repo as `notebooks/03_train_eval.ipynb` and unzip into `models/` + `docs/`.

In [ ]:
import shutil
import zipfile

art = Path('/content/artifacts')
art.mkdir(exist_ok=True)
shutil.copy('/content/best.pt', art / 'best.pt')
shutil.copy('/content/metrics.json', art / 'metrics.json')
for src, dst_name in ((RUN_A / 'results.csv', 'run_a_results.csv'), (RUN_B / 'results.csv', 'run_b_results.csv'),
                      (RUN_A / 'args.yaml', 'run_a_args.yaml'), (RUN_B / 'args.yaml', 'run_b_args.yaml')):
    if Path(src).exists():
        shutil.copy(src, art / dst_name)
for png in save_dir.glob('*.png'):
    shutil.copy(png, art / f'val_{png.name}')
zpath = shutil.make_archive('/content/capstone_artifacts', 'zip', art)
print('artifacts zip:', zpath)
for f in sorted(art.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.0f} KB)')